In [3]:
import os
import math
import warnings
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import SlabGenerator

warnings.filterwarnings("ignore")

# --- CONFIGURATION ---
structures_dir = "Na_Na3SbS4_Data/structures"
output_dir = "Na_Na3SbS4_Data/Interface_MD_Manual"
target_z_length = 20.0
min_xy_length = 1.0

SCAN_MILLERS = [
    (0, 0, 1)
]

def find_structure_by_formula(folder, formula):
    for fname in os.listdir(folder):
        if fname.endswith("vasp") and formula in fname:
            return os.path.join(folder, fname)
    return None

def get_standard_structure(filepath):
    try:
        s = Structure.from_file(filepath)
        sga = SpacegroupAnalyzer(s)
        conv = sga.get_conventional_standard_structure()
        print(f"Loaded {os.path.basename(filepath)} -> Conventional ({len(conv)} atoms)")
        return conv
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def get_slab_from_bulk(bulk, miller):
    slabgen = SlabGenerator(bulk, miller, 10.0, 10.0, lll_reduce=True)
    return slabgen.get_slab()

def find_optimal_scaling(a1, a2, min_L, max_L):
    best_mismatch = float('inf')
    best_n = (1, 1)
    
    for n1 in range(1, 5):
        L1 = n1 * a1
        if L1 < min_L: continue
        if L1 > max_L: break
        
        for n2 in range(1, 5):
            L2 = n2 * a2
            if L2 < min_L: continue
            if L2 > max_L: break
            
            mismatch = abs(L1 - L2) / max(L1, L2)
            if mismatch < best_mismatch:
                best_mismatch = mismatch
                best_n = (n1, n2)
                
    return best_n, best_mismatch

def manual_stack_001_optimized(s1, s2):
    print("\n--- Building (001) Interface with Optimal XY Scaling ---")
    
    slab1 = s1
    slab2 = s2
    
    a1, b1 = slab1.lattice.a, slab1.lattice.b
    a2, b2 = slab2.lattice.a, slab2.lattice.b
    
    (na1, na2), miss_a = find_optimal_scaling(a1, a2, min_xy_length, 60.0)
    (nb1, nb2), miss_b = find_optimal_scaling(b1, b2, min_xy_length, 60.0)
    
    avg_mismatch = (miss_a + miss_b) / 2
    
    sc1 = [na1, nb1]
    sc2 = [na2, nb2]
    
    print(f"Scaling LiGa (001): {sc1} | Li2Ga (001): {sc2}")
    print(f"Mismatch: {avg_mismatch:.2%}")

    # Z-Supercells
    n_z1 = math.ceil((target_z_length/2) / slab1.lattice.c)
    n_z2 = math.ceil((target_z_length/2) / slab2.lattice.c)
    
    # Ensure at least 2 unit cells thick to avoid single-layer artifacts
    n_z1 = max(n_z1, 2)
    n_z2 = max(n_z2, 2)
    
    print(f"Layers Z: LiGa={n_z1}, Li2Ga={n_z2}")
    
    slab1.make_supercell([sc1[0], sc1[1], n_z1])
    slab2.make_supercell([sc2[0], sc2[1], n_z2])
    
    # Coordinates
    coords1 = slab1.cart_coords
    coords2 = slab2.cart_coords
    species1 = slab1.species
    species2 = slab2.species

    # Normalize Z
    z1_min = np.min(coords1[:, 2])
    coords1[:, 2] -= z1_min
    z1_max = np.max(coords1[:, 2])
    
    z2_min = np.min(coords2[:, 2])
    z2_max = np.max(coords2[:, 2])
    
    # REDUCED GAP to 1.8 A (closer to bond length)
    interface_gap = 2
    z_offset = z1_max + interface_gap
    
    coords2[:, 2] -= z2_min
    coords2[:, 2] += z_offset
    
    # Final Box
    final_a = slab1.lattice.a
    final_b = slab1.lattice.b
    
    # IMPORTANT: The total C must include the gap at the TOP boundary too
    # otherwise atoms wrap around and crash into each other
    final_c = z_offset + (z2_max - z2_min) + interface_gap 
    
    new_lattice = Lattice.from_parameters(final_a, final_b, final_c, 90, 90, 90)
    
    combined_species = list(species1) + list(species2)
    combined_coords = np.vstack([coords1, coords2])
    
    # XY Center
    L1_a, L1_b = final_a, final_b
    L2_a = slab2.lattice.a 
    L2_b = slab2.lattice.b
    
    shift_x = (L1_a - L2_a) / 2.0
    shift_y = (L1_b - L2_b) / 2.0
    
    combined_coords[len(species1):, 0] += shift_x
    combined_coords[len(species1):, 1] += shift_y
    
    struct = Structure(new_lattice, combined_species, combined_coords, coords_are_cartesian=True)
    struct = struct.get_sorted_structure()
    
    print(f"Final Stack: {len(struct)} atoms. Box: {new_lattice.abc}")
    return struct

def create_thick_slab_interface():
    print("Searching for LiGa and Li2Ga structures...")
    f1 = Structure.from_file("Na_Na3SbS4_Data/structures/Na_mp-127.vasp")
    # find_structure_by_formula(structures_dir, "Na_mp")
    f2 = find_structure_by_formula(structures_dir, "Na3SbS4")
    if not f1 or not f2:
        print("Error: Missing structure files.")
        return None

    s1 = SpacegroupAnalyzer(f1).get_conventional_standard_structure()
    s2 = get_standard_structure(f2)
    if not s1 or not s2: return None

    return manual_stack_001_optimized(s1, s2)

# --- MAIN ---
os.makedirs(output_dir, exist_ok=True)
struct = create_thick_slab_interface()

if struct:
    ld = LammpsData.from_structure(struct, atom_style="atomic")
    ld.write_file(os.path.join(output_dir, "data.lammps"))
    print(f"\nSuccess! Data written to {os.path.join(output_dir, 'data.lammps')}")
    
    els = sorted(list(struct.composition.get_el_amt_dict().keys()))
    print("Atom Type Mapping:")
    for i, e in enumerate(els):
        print(f"  Type {i+1} = {e}")
else:
    print("Critical Error: Could not generate any structure.")

Searching for LiGa and Li2Ga structures...


NameError: name 'f1' is not defined